# Human-in-the-loop

[Middleware](https://docs.langchain.com/oss/python/langchain/middleware/built-in#human-in-the-loop) Human-in-the-Loop (HITL) cho phép bạn bổ sung sự giám sát của con người vào các tool call của agent.
Khi model đề xuất một hành động có thể cần được xem xét, chẳng hạn ghi vào tệp hoặc thực thi SQL, middleware có thể tạm dừng quá trình thực thi và chờ quyết định.

Middleware thực hiện việc này bằng cách kiểm tra từng tool call dựa trên một policy có thể cấu hình. Nếu cần can thiệp, middleware sẽ phát ra một [interrupt](https://reference.langchain.com/python/langgraph/types/interrupt) để dừng quá trình thực thi. Graph state được lưu thông qua [persistence layer](https://docs.langchain.com/oss/python/langgraph/persistence) của LangGraph, nhờ đó quá trình thực thi có thể tạm dừng an toàn và tiếp tục sau đó.

Quyết định của con người sẽ quyết định bước tiếp theo: hành động có thể được phê duyệt nguyên trạng (`approve`), được chỉnh sửa trước khi chạy (`edit`), bị từ chối kèm phản hồi (`reject`), hoặc được trả lời trực tiếp (`respond`) đối với các tool kiểu "ask user" (hỏi người dùng).

## Các loại quyết định khi interrupt

[Middleware](https://docs.langchain.com/oss/python/langchain/middleware/built-in#human-in-the-loop) định nghĩa bốn cách có sẵn để con người phản hồi một interrupt:

| Loại quyết định | Mô tả                                                                                                                                | Ví dụ sử dụng                                     |
| --------------- | ------------------------------------------------------------------------------------------------------------------------------------ | ------------------------------------------------- |
| ✅ `approve`     | Thực thi tool với các argument gốc do agent đề xuất.                                                                                 | Gửi bản nháp email đúng như đã soạn               |
| ✏️ `edit`       | Chỉnh sửa các argument của tool trước khi thực thi.                                                                                  | Đổi người nhận trước khi gửi email                |
| ❌ `reject`      | Bỏ qua hoàn toàn việc thực thi tool call này và trả phản hồi từ chối về cho agent.                                                    | Từ chối việc xóa tệp và giải thích lý do          |
| 💬 `respond`    | Trả trực tiếp tin nhắn của con người dưới dạng một tool result tổng hợp, bỏ qua việc thực thi, dành cho các tool kiểu "ask user". | Trả lời một prompt `"ask_user"` bằng câu trả lời trực tiếp |

Các loại quyết định khả dụng cho từng tool phụ thuộc vào policy bạn cấu hình trong `interrupt_on`.
Khi nhiều tool call bị tạm dừng cùng lúc, mỗi hành động cần một quyết định riêng.
Các quyết định phải được cung cấp theo đúng thứ tự xuất hiện của các hành động trong interrupt request.

Hãy dùng `reject` khi con người từ chối hành động được yêu cầu. Chỉ dùng `respond` khi con người đóng vai trò của chính tool đó, chẳng hạn trả lời một prompt `ask_user`. Không dùng `respond` để từ chối các tool có side effect, vì tin nhắn của nó được coi là một tool result thành công.

<div class="alert alert-success">

Khi **chỉnh sửa** các argument của tool, hãy thay đổi một cách thận trọng. Những thay đổi lớn so với argument gốc có thể khiến model đánh giá lại cách tiếp cận của nó, dẫn đến việc thực thi tool nhiều lần hoặc thực hiện các hành động ngoài dự kiến.

</div>

## Cấu hình interrupt

Để sử dụng HITL, hãy thêm [middleware](https://docs.langchain.com/oss/python/langchain/middleware/built-in#human-in-the-loop) vào danh sách `middleware` của agent khi tạo agent.

Bạn cấu hình middleware bằng một mapping giữa các tool action và những loại quyết định được phép cho từng action. Middleware sẽ ngắt quá trình thực thi khi một tool call khớp với một action trong mapping.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


agent = create_agent(
    model="gpt-5.5",
    tools=[write_file, execute_sql, read_data],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "write_file": True,  # Cho phép mọi quyết định (approve, edit, reject, respond)
                "execute_sql": {"allowed_decisions": ["approve", "reject"]},  # Không cho phép chỉnh sửa
                "read_data": False, # Thao tác an toàn, không cần phê duyệt
            },
            # Tiền tố cho thông điệp interrupt - được ghép với tên tool và args để tạo thành thông điệp đầy đủ
            # ví dụ: "Thực thi tool đang chờ phê duyệt: execute_sql với query='DELETE FROM...'"
            # Từng tool có thể ghi đè giá trị này bằng cách chỉ định "description" trong cấu hình interrupt của tool đó
            description_prefix="Thực thi tool đang chờ phê duyệt",
        ),
    ],
    # Human-in-the-loop yêu cầu checkpointing để xử lý các interrupt.
    # Trong môi trường production, hãy dùng checkpointer bền vững như AsyncPostgresSaver hoặc MongoDBSaver.
    checkpointer=InMemorySaver(),
)

<div class="alert alert-info">

Bạn phải cấu hình một checkpointer để lưu giữ graph state xuyên suốt các interrupt.
Trong môi trường production, hãy dùng checkpointer bền vững như [`AsyncPostgresSaver`](https://reference.langchain.com/python/langgraph/checkpoints/#langgraph.checkpoint.postgres.aio.AsyncPostgresSaver) hoặc [`MongoDBSaver`](https://pypi.org/project/langgraph-checkpoint-mongodb/). Với mục đích kiểm thử hoặc tạo prototype, hãy dùng [`InMemorySaver`](https://reference.langchain.com/python/langgraph/checkpoints/#langgraph.checkpoint.memory.InMemorySaver).

Khi gọi agent, hãy truyền vào một `config` chứa **thread ID** để liên kết quá trình thực thi với một luồng hội thoại.
Xem [tài liệu về interrupt của LangGraph](https://docs.langchain.com/oss/python/langgraph/interrupts) để biết thêm chi tiết.

</div>

**Tùy chọn cấu hình**

* `interrupt_on` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">dict</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">required</span>
  
  Mapping giữa tên tool và cấu hình phê duyệt. Giá trị có thể là `True` (interrupt với cấu hình mặc định), `False` (tự động phê duyệt), hoặc một đối tượng `InterruptOnConfig`.

* `description_prefix` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">string</span> <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">default:"Tool execution requires approval"</span>
  
  Tiền tố cho phần mô tả của các action request

  **Các tùy chọn của `InterruptOnConfig`:**

* `allowed_decisions` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">list[string]</span>

  Danh sách các quyết định được phép: `'approve'`, `'edit'`, `'reject'` hoặc `'respond'`

* `description` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">string | callable</span>

  Chuỗi tĩnh hoặc hàm callable để tùy chỉnh phần mô tả

* `when` <span style="background-color: #f3f4f6; color: #4b5563; padding: 2px 8px; border-radius: 6px; font-size: 0.85em; font-family: monospace;">callable</span>

  Predicate tùy chọn nhận vào một [ToolCallRequest](https://reference.langchain.com/python/langgraph.prebuilt/tool_node/ToolCallRequest) và trả về `True` để interrupt hoặc `False` để tự động phê duyệt. Dùng predicate này để quyết định có interrupt hay không dựa trên các argument của lệnh gọi. Yêu cầu `langchain>=1.3.3`.

## Interrupt có điều kiện

Theo mặc định, mọi tool call được liệt kê trong `interrupt_on` đều tạm dừng để chờ xem xét. Để chỉ tạm dừng một số lệnh gọi, hãy thêm predicate `when` vào `InterruptOnConfig` của tool. Predicate nhận một `ToolCallRequest` và trả về `True` để interrupt hoặc `False` để tự động phê duyệt, nhờ đó bạn có thể quyết định dựa trên các argument của tool.

<div class="alert alert-info">

Interrupt có điều kiện yêu cầu `langchain>=1.3.3`.

</div>

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware, ToolCallRequest
from langgraph.checkpoint.memory import InMemorySaver


def writes_outside_workspace(request: ToolCallRequest) -> bool:
    """Tạm dừng các thao tác ghi vào đường dẫn nằm ngoài thư mục workspace."""
    path = request.tool_call["args"].get("path", "")
    return not path.startswith("/workspace/")


def is_write_query(request: ToolCallRequest) -> bool:
    """Tạm dừng các câu lệnh SQL không phải là SELECT chỉ đọc."""
    query = request.tool_call["args"].get("query", "")
    return not query.lstrip().upper().startswith("SELECT")


agent = create_agent(
    model="gpt-5.5",
    tools=[write_file, execute_sql, read_data],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "write_file": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                    "when": writes_outside_workspace,
                },
                "execute_sql": {
                    "allowed_decisions": ["approve", "reject"],
                    "when": is_write_query,
                },
            },
        ),
    ],
    checkpointer=InMemorySaver(),
)

Khi predicate `when` trả về `False`, lệnh gọi sẽ chạy mà không bị interrupt. Khi nó trả về `True`, hoặc khi bạn bỏ qua `when`, lệnh gọi sẽ tạm dừng như thông thường. Các lệnh gọi có kết quả đánh giá là `False` sẽ không bao giờ được đưa vào interrupt batch, do đó người review chỉ thấy những action cần đưa ra quyết định.

## Phản hồi các interrupt

Khi bạn gọi agent, agent sẽ chạy cho đến khi hoàn tất hoặc gặp một interrupt. Interrupt được kích hoạt khi một tool call khớp với policy bạn đã cấu hình trong `interrupt_on`. Với `version="v2"`, kết quả trả về là một `GraphOutput` có thuộc tính `interrupts` chứa các action cần được xem xét. Sau đó, bạn có thể hiển thị các action này cho người review và tiếp tục quá trình thực thi khi đã có quyết định.

In [ ]:
from langgraph.types import Command

# Human-in-the-loop tận dụng persistence layer của LangGraph.
# Bạn phải cung cấp một thread ID để liên kết quá trình thực thi với một luồng hội thoại,
# nhờ đó cuộc hội thoại có thể được tạm dừng và tiếp tục (như cần thiết cho việc review của con người).
config = {"configurable": {"thread_id": "some_id"}}
# Chạy graph cho đến khi gặp interrupt.
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Xóa các bản ghi cũ khỏi cơ sở dữ liệu",
            }
        ]
    },
    config=config,
    version="v2",
)

# result là một GraphOutput có .value và .interrupts
print(result.interrupts)
# > (
# >    Interrupt(
# >       value={
# >          'action_requests': [
# >             {
# >                'name': 'execute_sql',
# >                'arguments': {'query': 'DELETE FROM records WHERE created_at < NOW() - INTERVAL \'30 days\';'},
# >                'description': 'Thực thi tool đang chờ phê duyệt\n\nTool: execute_sql\nArgs: {...}'
# >             }
# >          ],
# >          'review_configs': [
# >             {
# >                'action_name': 'execute_sql',
# >                'allowed_decisions': ['approve', 'reject']
# >             }
# >          ]
# >       }
# >    ),
# > )


# Tiếp tục với quyết định phê duyệt
agent.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}  # hoặc "reject"
    ),
    config=config, # Dùng cùng thread ID để tiếp tục cuộc hội thoại đang tạm dừng
    version="v2",
)

### Các loại quyết định

✅ **approve**
    
Dùng `approve` để phê duyệt tool call nguyên trạng và thực thi mà không thay đổi gì.

In [ ]:
agent.invoke(
    Command(
        # Các quyết định được cung cấp dưới dạng danh sách, mỗi action đang được review có một quyết định.
        # Thứ tự của các quyết định phải khớp với thứ tự của các action
        # trong interrupt request.
        resume={
            "decisions": [
                {
                    "type": "approve",
                }
            ]
        }
    ),
    config=config,  # Dùng cùng thread ID để tiếp tục cuộc hội thoại đang tạm dừng
    version="v2",
)

✏️ **edit**

Dùng `edit` để chỉnh sửa tool call trước khi thực thi. Hãy cung cấp action đã chỉnh sửa cùng với tên tool và các argument mới.

In [ ]:
agent.invoke(
    Command(
        # Các quyết định được cung cấp dưới dạng danh sách, mỗi action đang được review có một quyết định.
        # Thứ tự của các quyết định phải khớp với thứ tự của các action
        # trong interrupt request.
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Action đã chỉnh sửa gồm tên tool và args
                    "edited_action": {
                        # Tên tool cần gọi.
                        # Thường sẽ giống với action ban đầu.
                        "name": "new_tool_name",
                        # Các argument truyền vào tool.
                        "args": {"key1": "new_value", "key2": "original_value"},
                    }
                }
            ]
        }
    ),
    config=config,  # Dùng cùng thread ID để tiếp tục cuộc hội thoại đang tạm dừng
    version="v2",
)

<div class="alert alert-success">

Khi **chỉnh sửa** các argument của tool, hãy thay đổi một cách thận trọng. Những thay đổi lớn so với argument gốc có thể khiến model đánh giá lại cách tiếp cận của nó, dẫn đến việc thực thi tool nhiều lần hoặc thực hiện các hành động ngoài dự kiến.

</div>

❌ **reject**

Dùng `reject` để từ chối tool call và cung cấp phản hồi thay cho việc thực thi. Tool sẽ không được thực thi.

In [ ]:
agent.invoke(
    Command(
        # Các quyết định được cung cấp dưới dạng danh sách, mỗi action đang được review có một quyết định.
        # Thứ tự của các quyết định phải khớp với thứ tự của các action
        # trong interrupt request.
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # Tùy chọn: giải thích lý do action bị từ chối
                    # và cho biết agent có nên thử lại bằng cách tiếp cận khác hay không.
                    "message": "Người dùng đã từ chối hành động này. Không thử lại lệnh gọi tool này.",
                }
            ]
        }
    ),
    config=config,  # Dùng cùng thread ID để tiếp tục cuộc hội thoại đang tạm dừng
    version="v2",
)

`message` được thêm vào cuộc hội thoại như một phản hồi giúp agent hiểu vì sao action bị từ chối và nên làm gì tiếp theo. Khi bạn bỏ qua `message`, middleware sẽ dùng một thông điệp từ chối mặc định, cho model biết rằng tool chưa được thực thi và không được thử lại cùng tool call đó trừ khi người dùng yêu cầu. Với các tool có side effect, hãy cung cấp một thông điệp riêng theo ngữ cảnh nghiệp vụ, nêu rõ agent nên từ bỏ hành động, đặt thêm câu hỏi, hay thử một phương án thay thế an toàn hơn.

💬 **respond**

Dùng `respond` cho các tool kiểu "ask user" mà phần triển khai thực sự của tool chính là câu trả lời của con người. Nội dung `message` được trả về trực tiếp dưới dạng tool result; bản thân tool không được thực thi.

In [ ]:
agent.invoke(
    Command(
        # Các quyết định được cung cấp dưới dạng danh sách, mỗi action đang được review có một quyết định.
        # Thứ tự của các quyết định phải khớp với thứ tự của các action
        # trong interrupt request.
        resume={
            "decisions": [
                {
                    "type": "respond",
                    # Câu trả lời của con người, được trả về trực tiếp dưới dạng tool result
                    "message": "Màu xanh dương.",
                }
            ]
        }
    ),
    config=config,  # Dùng cùng thread ID để tiếp tục cuộc hội thoại đang tạm dừng
    version="v2",
)

`message` được trả về cho agent dưới dạng một `ToolMessage` thành công. Hãy dùng `respond` khi tool được thiết kế có chủ đích như một placeholder cho đầu vào của con người, ví dụ một tool `ask_user` dùng để hỏi làm rõ thông tin. Không dùng `respond` để từ chối một hành động được đề xuất, vì nó báo cho model rằng tool đã hoàn thành thành công.

### Nhiều quyết định

Khi có nhiều action đang được review, hãy cung cấp một quyết định cho mỗi action, theo đúng thứ tự xuất hiện của chúng trong interrupt:

In [ ]:
{
    "decisions": [
        {"type": "approve"},
        {
            "type": "edit",
            "edited_action": {
                "name": "tool_name",
                "args": {"param": "new_value"}
            }
        },
        {
            "type": "reject",
            "message": "Hành động này không được phép"
        }
    ]
}

## Streaming với human-in-the-loop

Bạn có thể stream các cập nhật theo thời gian thực trong khi agent chạy và xử lý interrupt bằng `stream_events()`. Dùng `stream.messages` để stream các token của LLM và `stream.values` để kiểm tra các snapshot trạng thái của agent nhằm phát hiện interrupt.

In [ ]:
from langgraph.types import Command

config = {"configurable": {"thread_id": "some_id"}}

# Stream tiến trình của agent và các token LLM cho đến khi gặp interrupt
stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "Xóa các bản ghi cũ khỏi cơ sở dữ liệu"}]},
    config=config,
    version="v3",
)
for message in stream.messages:
    for token in message.text:
        print(token, end="", flush=True)

# Kiểm tra xem lượt chạy có tạm dừng để chờ đầu vào của con người hay không
if stream.interrupted:
    print(f"\n\nInterrupt: {stream.interrupts}")

# Tiếp tục với streaming sau khi con người đưa ra quyết định
stream = agent.stream_events(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config,
    version="v3",
)
for message in stream.messages:
    for token in message.text:
        print(token, end="", flush=True)

Xem hướng dẫn [Streaming](https://docs.langchain.com/oss/python/langchain/streaming) để biết thêm chi tiết về các stream mode.

## Vòng đời thực thi

Middleware định nghĩa một hook `after_model` chạy sau khi model tạo ra phản hồi nhưng trước khi bất kỳ tool call nào được thực thi:

1. Agent gọi model để tạo phản hồi.
2. Middleware kiểm tra phản hồi để tìm các tool call.
3. Nếu có lệnh gọi nào cần đầu vào của con người, middleware sẽ xây dựng một [HITLRequest](https://reference.langchain.com/python/langchain/agents/middleware/human_in_the_loop/HITLRequest) gồm `action_requests` và `review_configs`, rồi gọi [interrupt](https://reference.langchain.com/python/langgraph/types/interrupt).
4. Agent chờ các quyết định của con người.
5. Dựa trên các quyết định trong `HITLResponse`, middleware thực thi các lệnh gọi đã được phê duyệt hoặc chỉnh sửa, tạo [ToolMessage](https://reference.langchain.com/python/langchain-core/messages/tool/ToolMessage) tổng hợp cho các lệnh gọi bị từ chối, trả trực tiếp phản hồi của con người dưới dạng [ToolMessage](https://reference.langchain.com/python/langchain-core/messages/tool/ToolMessage) cho các quyết định `respond`, rồi tiếp tục quá trình thực thi.

## Logic HITL tùy chỉnh

Với các workflow chuyên biệt hơn, bạn có thể tự xây dựng logic HITL tùy chỉnh trực tiếp bằng primitive [interrupt](https://reference.langchain.com/python/langgraph/types/interrupt) và abstraction [middleware](https://docs.langchain.com/oss/python/langchain/middleware).

Hãy xem lại [vòng đời thực thi](https://docs.langchain.com/oss/python/langchain/human-in-the-loop#execution-lifecycle) ở trên để hiểu cách tích hợp interrupt vào hoạt động của agent.